# GoEmotions Processed Dataset EDA

This notebook validates the finalized 3-class GoEmotions sentiment splits kept for DistilBERT training.

It loads the processed files from `../data/datasets/gomotions_processed/` and performs concise, modeling-focused checks:
- dataset sizes
- label distribution
- class balance
- sample rows per class
- text length statistics
- duplicate and empty-row checks
- train/test overlap checks

Files used:
- `goemotions_train.csv`
- `goemotions_test.csv`

Each processed file is expected to contain only:
- `text`
- `label` (`positive`, `neutral`, `negative`)

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

PROCESSED_DIR = Path("../data/datasets/gomotions_processed")
TRAIN_PATH = PROCESSED_DIR / "goemotions_train.csv"
TEST_PATH = PROCESSED_DIR / "goemotions_test.csv"
EXPECTED_COLUMNS = ["text", "label"]
SENTIMENT_ORDER = ["positive", "neutral", "negative"]

In [2]:
def load_processed_split(path: Path, split_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing processed {split_name} split: {path}")

    df = pd.read_csv(path)
    if list(df.columns) != EXPECTED_COLUMNS:
        raise ValueError(
            f"Unexpected columns in {split_name} split. "
            f"Expected {EXPECTED_COLUMNS}, found {list(df.columns)}"
        )

    invalid_labels = sorted(set(df["label"].dropna().astype(str)) - set(SENTIMENT_ORDER))
    if invalid_labels:
        raise ValueError(f"Unexpected labels in {split_name} split: {invalid_labels}")

    return df


def class_distribution(df: pd.DataFrame) -> pd.DataFrame:
    counts = df["label"].value_counts().reindex(SENTIMENT_ORDER, fill_value=0)
    percentages = (counts / len(df) * 100).round(2) if len(df) else counts.astype(float)
    return pd.DataFrame({"count": counts, "percent": percentages})


def text_length_stats(df: pd.DataFrame) -> pd.Series:
    word_counts = df["text"].astype(str).str.split().map(len)
    char_counts = df["text"].astype(str).str.len()
    return pd.Series(
        {
            "rows": int(len(df)),
            "mean_words": round(float(word_counts.mean()), 2),
            "median_words": float(word_counts.median()),
            "p90_words": float(word_counts.quantile(0.90)),
            "max_words": int(word_counts.max()),
            "mean_chars": round(float(char_counts.mean()), 2),
            "median_chars": float(char_counts.median()),
            "p90_chars": float(char_counts.quantile(0.90)),
            "max_chars": int(char_counts.max()),
        }
    )

In [3]:
def sample_rows_by_class(df: pd.DataFrame, n: int = 3) -> pd.DataFrame:
    samples = []
    for label in SENTIMENT_ORDER:
        class_rows = df.loc[df["label"] == label, ["label", "text"]].head(n)
        samples.append(class_rows)
    return pd.concat(samples, ignore_index=True)


def split_balance_summary(df: pd.DataFrame, split_name: str) -> dict[str, object]:
    counts = df["label"].value_counts().reindex(SENTIMENT_ORDER, fill_value=0)
    imbalance_ratio = round(float(counts.max() / counts.min()), 2) if counts.min() > 0 else None
    return {
        "split": split_name,
        "rows": int(len(df)),
        "largest_class": counts.idxmax(),
        "largest_class_count": int(counts.max()),
        "smallest_class": counts.idxmin(),
        "smallest_class_count": int(counts.min()),
        "imbalance_ratio": imbalance_ratio,
    }


def confirm_quality(df: pd.DataFrame) -> dict[str, int]:
    return {
        "empty_rows": int(df["text"].fillna("").astype(str).str.strip().eq("").sum()),
        "duplicate_text_rows": int(df.duplicated(subset=["text"]).sum()),
        "duplicate_text_label_rows": int(df.duplicated(subset=["text", "label"]).sum()),
    }

In [4]:
train_df = load_processed_split(TRAIN_PATH, "train")
test_df = load_processed_split(TEST_PATH, "test")

processed_overview = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "columns": len(train_df.columns),
            "path": str(TRAIN_PATH),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "columns": len(test_df.columns),
            "path": str(TEST_PATH),
        },
    ]
)

print("Processed GoEmotions files")
display(processed_overview)

Processed GoEmotions files


,split,rows,columns,path
0,train,33344,2,../data/datasets/gomotions_processed/goemotions_tra...
1,test,13465,2,../data/datasets/gomotions_processed/goemotions_tes...


In [5]:
schema_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "columns": list(train_df.columns),
            "matches_expected_schema": list(train_df.columns) == EXPECTED_COLUMNS,
        },
        {
            "split": "test",
            "columns": list(test_df.columns),
            "matches_expected_schema": list(test_df.columns) == EXPECTED_COLUMNS,
        },
    ]
)

print("Schema validation")
display(schema_summary)

Schema validation


,split,columns,matches_expected_schema
0,train,"[text, label]",True
1,test,"[text, label]",True


In [6]:
dataset_sizes = pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows": [len(train_df), len(test_df)],
    }
)
train_distribution = class_distribution(train_df)
test_distribution = class_distribution(test_df)
balance_summary = pd.DataFrame(
    [
        split_balance_summary(train_df, "train"),
        split_balance_summary(test_df, "test"),
    ]
)

print("Dataset sizes")
display(dataset_sizes)

print("Train label distribution")
display(train_distribution)

print("Test label distribution")
display(test_distribution)

print("Class balance summary")
display(balance_summary)

Dataset sizes


,split,rows
0,train,33344
1,test,13465


Train label distribution


,count,percent
label,,
positive,12854,38.55
neutral,12241,36.71
negative,8249,24.74


Test label distribution


,count,percent
label,,
positive,4582,34.03
neutral,5360,39.81
negative,3523,26.16


Class balance summary


,split,rows,largest_class,largest_class_count,smallest_class,smallest_class_count,imbalance_ratio
0,train,33344,positive,12854,negative,8249,1.56
1,test,13465,neutral,5360,negative,3523,1.52


In [7]:
print("Sample train rows per class")
display(sample_rows_by_class(train_df, n=3))

print("Sample test rows per class")
display(sample_rows_by_class(test_df, n=3))

Sample train rows per class


,label,text
0,positive,Man I love reddit.
1,positive,Right? Considering it’s such an important docu...
2,positive,"I appreciate it, that's good to know. I hope I..."
3,neutral,"[NAME] was nowhere near them, he was by the Fa..."
4,neutral,"I have, and now that you mention it, I think t..."
5,neutral,BUT IT'S HER TURN! /s
6,negative,That game hurt.
7,negative,For extra measure tape it right by your crotch...
8,negative,"Dark and funny, but not really nice guy. He ha..."


Sample test rows per class


,label,text
0,positive,Happy cake day u/sneakpeekbot!
1,positive,[NAME] finally getting WCW going I see..
2,positive,"Yes!!!! Great weekend. As others have said, I ..."
3,neutral,Just take them and get them vaccinated without...
4,neutral,If you’ll check my medical records you’ll see ...
5,neutral,"Dating [NAME] tends to wear you down, late 20s..."
6,negative,Both parents were drunk. Where were the small ...
7,negative,"yeah [NAME] can eat glass, that guy fuckin sucks"
8,negative,>CFMEU Any excuse to act like thugs and not do...


In [8]:
length_summary = pd.DataFrame(
    {
        "train": text_length_stats(train_df),
        "test": text_length_stats(test_df),
    }
)
quality_summary = pd.DataFrame(
    [
        {"split": "train", **confirm_quality(train_df)},
        {"split": "test", **confirm_quality(test_df)},
        {
            "split": "cross_split_overlap",
            "empty_rows": 0,
            "duplicate_text_rows": len(set(train_df["text"]).intersection(set(test_df["text"]))),
            "duplicate_text_label_rows": 0,
        },
    ]
)

print("Text length summary")
display(length_summary)

print("Quality checks")
display(quality_summary)

assert quality_summary.loc[quality_summary["split"] == "train", "empty_rows"].item() == 0
assert quality_summary.loc[quality_summary["split"] == "test", "empty_rows"].item() == 0
assert quality_summary.loc[quality_summary["split"] == "train", "duplicate_text_rows"].item() == 0
assert quality_summary.loc[quality_summary["split"] == "test", "duplicate_text_rows"].item() == 0
assert quality_summary.loc[quality_summary["split"] == "cross_split_overlap", "duplicate_text_rows"].item() == 0

Text length summary


,train,test
rows,33344.00,13465.00
mean_words,12.64,13.25
median_words,12.00,13.00
p90_words,22.00,23.00
max_words,32.00,33.00
mean_chars,67.32,70.63
median_chars,64.00,68.00
p90_chars,119.00,121.00
max_chars,703.00,542.00


Quality checks


,split,empty_rows,duplicate_text_rows,duplicate_text_label_rows
0,train,0,0,0
1,test,0,0,0
2,cross_split_overlap,0,0,0


In [9]:
summary_lines = [
    "GOEMOTIONS_DATASET_SUMMARY",
    "",
    f"train size: {len(train_df)}",
    f"test size: {len(test_df)}",
    f"train class distribution: {train_df['label'].value_counts().reindex(SENTIMENT_ORDER, fill_value=0).to_dict()}",
    f"test class distribution: {test_df['label'].value_counts().reindex(SENTIMENT_ORDER, fill_value=0).to_dict()}",
    f"train duplicates remaining: {confirm_quality(train_df)['duplicate_text_rows']}",
    f"test duplicates remaining: {confirm_quality(test_df)['duplicate_text_rows']}",
    f"train empty rows remaining: {confirm_quality(train_df)['empty_rows']}",
    f"test empty rows remaining: {confirm_quality(test_df)['empty_rows']}",
    f"cross-split overlap remaining: {len(set(train_df['text']).intersection(set(test_df['text'])))}",
    f"schema valid for both splits: {list(train_df.columns) == EXPECTED_COLUMNS and list(test_df.columns) == EXPECTED_COLUMNS}",
]

print("\n".join(summary_lines))

GOEMOTIONS_DATASET_SUMMARY

train size: 33344
test size: 13465
train class distribution: {'positive': 12854, 'neutral': 12241, 'negative': 8249}
test class distribution: {'positive': 4582, 'neutral': 5360, 'negative': 3523}
train duplicates remaining: 0
test duplicates remaining: 0
train empty rows remaining: 0
test empty rows remaining: 0
cross-split overlap remaining: 0
schema valid for both splits: True


## ERROR_ANALYSIS_WRONG_PREDICTIONS

This section inspects the saved DistilBERT test predictions and focuses only on model mistakes so we can understand where the classifier is failing.

In [10]:
PREDICTIONS_PATH = Path("../artifacts/reports/distilbert_goemotions/distilbert_goemotions_test_predictions.csv")
REQUIRED_PREDICTION_COLUMNS = ["text", "true_label", "predicted_label"]
PROBABILITY_COLUMNS = ["prob_negative", "prob_neutral", "prob_positive"]

if not PREDICTIONS_PATH.exists():
    raise FileNotFoundError(f"Missing predictions file: {PREDICTIONS_PATH}")

prediction_df = pd.read_csv(PREDICTIONS_PATH)
missing_prediction_columns = [
    column for column in REQUIRED_PREDICTION_COLUMNS if column not in prediction_df.columns
]
if missing_prediction_columns:
    raise ValueError(
        f"Missing required prediction columns: {missing_prediction_columns}. "
        f"Found columns: {list(prediction_df.columns)}"
    )

prediction_df["text"] = prediction_df["text"].fillna("").astype(str).str.strip()
prediction_df["true_label"] = prediction_df["true_label"].fillna("").astype(str).str.strip().str.lower()
prediction_df["predicted_label"] = prediction_df["predicted_label"].fillna("").astype(str).str.strip().str.lower()
prediction_df = prediction_df[prediction_df["text"] != ""].reset_index(drop=True)
prediction_df["text_len"] = prediction_df["text"].apply(lambda value: len(str(value).split()))

wrong_df = prediction_df[prediction_df["true_label"] != prediction_df["predicted_label"]].copy()
correct_df = prediction_df[prediction_df["true_label"] == prediction_df["predicted_label"]].copy()
error_rate = len(wrong_df) / len(prediction_df) if len(prediction_df) else 0.0

print("Loaded predictions:", PREDICTIONS_PATH)
print(f"total rows: {len(prediction_df)}")
print(f"number of wrong predictions: {len(wrong_df)}")
print(f"error rate: {error_rate:.4f}")

Loaded predictions: ../artifacts/reports/distilbert_goemotions/distilbert_goemotions_test_predictions.csv
total rows: 13465
number of wrong predictions: 4192
error rate: 0.3113


In [11]:
errors_by_true = wrong_df["true_label"].value_counts().reindex(SENTIMENT_ORDER, fill_value=0)
errors_by_predicted = wrong_df["predicted_label"].value_counts().reindex(SENTIMENT_ORDER, fill_value=0)
error_crosstab = pd.crosstab(
    wrong_df["true_label"],
    wrong_df["predicted_label"],
).reindex(index=SENTIMENT_ORDER, columns=SENTIMENT_ORDER, fill_value=0)

print("Errors per TRUE label")
display(errors_by_true.to_frame(name="count"))

print("Errors per PREDICTED label")
display(errors_by_predicted.to_frame(name="count"))

print("True vs predicted errors")
display(error_crosstab)

Errors per TRUE label


,count
true_label,
positive,1347
neutral,1583
negative,1262


Errors per PREDICTED label


,count
predicted_label,
positive,842
neutral,2010
negative,1340


True vs predicted errors


predicted_label,positive,neutral,negative
true_label,,,
positive,0,963,384
neutral,627,0,956
negative,215,1047,0


In [12]:
def example_view(df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[:, ["text", "true_label", "predicted_label"]].rename(
        columns={
            "text": "TEXT",
            "true_label": "TRUE",
            "predicted_label": "PREDICTED",
        }
    )


def sample_confusion_examples(
    df: pd.DataFrame,
    *,
    true_label: str,
    predicted_label: str,
    n: int,
) -> pd.DataFrame:
    subset = df[
        (df["true_label"] == true_label)
        & (df["predicted_label"] == predicted_label)
    ].copy()
    if subset.empty:
        return subset
    return subset.head(min(n, len(subset)))


random_wrong_examples = wrong_df.sample(n=min(10, len(wrong_df)), random_state=42) if len(wrong_df) else wrong_df

print("10 random wrong predictions")
display(example_view(random_wrong_examples))

print("5 examples: negative -> neutral")
display(example_view(sample_confusion_examples(wrong_df, true_label="negative", predicted_label="neutral", n=5)))

print("5 examples: neutral -> positive")
display(example_view(sample_confusion_examples(wrong_df, true_label="neutral", predicted_label="positive", n=5)))

print("5 examples: positive -> neutral")
display(example_view(sample_confusion_examples(wrong_df, true_label="positive", predicted_label="neutral", n=5)))

10 random wrong predictions


,TEXT,TRUE,PREDICTED
8590,"""Everyone knows the [NAME] make good stuff!"" -...",neutral,positive
7541,"Wait, what does golf have to do with this?",negative,neutral
6777,Don't you're making me paranoid man,neutral,negative
3740,[NAME] shot has been quite frankly terrible to...,neutral,negative
3147,Hey we have feelings too nerd,negative,neutral
6969,She has major SB😖,negative,neutral
8642,It also looks like it left an indented welt. H...,neutral,negative
7955,"Nah, black bears are big scaredy cats Now brow...",negative,neutral
4136,"""A lot can change in 365 days. Hang in there.""",positive,neutral
6255,Proper us of cell phone while operating a moto...,neutral,positive


5 examples: negative -> neutral


,TEXT,TRUE,PREDICTED
4,>CFMEU Any excuse to act like thugs and not do...,negative,neutral
17,"But this was a supposed psychopath attack, whi...",negative,neutral
19,What was her emergency other than a teen throw...,negative,neutral
29,My thoughts EXACTLY [NAME] looks dead.,negative,neutral
38,If you're talking about tfue you're insane. He...,negative,neutral


5 examples: neutral -> positive


,TEXT,TRUE,PREDICTED
10,One of the low key best hikes in AZ! Better th...,neutral,positive
14,Better than showing weakness,neutral,positive
37,if it makes you feel better I still think your...,neutral,positive
89,"oh my, the edge. i bet he comes with a pre-sel...",neutral,positive
107,I look forward to him posing for photos with h...,neutral,positive


5 examples: positive -> neutral


,TEXT,TRUE,PREDICTED
13,[NAME] finally getting WCW going I see..,positive,neutral
32,This is accurate. Source: MA resident who work...,positive,neutral
56,"If that really meet to comfort us, guess who h...",positive,neutral
82,Well at least he wont be hungry ever again,positive,neutral
160,"Not only your boyfriend thinks you are hot, Ii...",positive,neutral


In [13]:
available_probability_columns = [
    column for column in PROBABILITY_COLUMNS if column in prediction_df.columns
]

if available_probability_columns:
    wrong_confidence_df = wrong_df.copy()
    wrong_confidence_df["predicted_confidence"] = wrong_confidence_df[available_probability_columns].max(axis=1)
    high_confidence_wrong_df = wrong_confidence_df.sort_values(
        "predicted_confidence", ascending=False
    ).head(10)

    print("Top 10 high-confidence wrong predictions")
    display(
        high_confidence_wrong_df[
            [
                "text",
                "true_label",
                "predicted_label",
                "predicted_confidence",
                *available_probability_columns,
            ]
        ].rename(
            columns={
                "text": "TEXT",
                "true_label": "TRUE",
                "predicted_label": "PREDICTED",
                "predicted_confidence": "PREDICTED_CONFIDENCE",
            }
        )
    )
else:
    high_confidence_wrong_df = pd.DataFrame()
    print("Probability columns not found, so high-confidence wrong prediction analysis was skipped.")

print("Wrong prediction text length summary")
display(wrong_df["text_len"].describe().to_frame(name="wrong_predictions"))

print("Correct prediction text length summary")
display(correct_df["text_len"].describe().to_frame(name="correct_predictions"))

Top 10 high-confidence wrong predictions


,TEXT,TRUE,PREDICTED,PREDICTED_CONFIDENCE,prob_negative,prob_neutral,prob_positive
2930,Thanks for further justifying my support for T...,neutral,positive,0.987232,0.003547,0.009220,0.987232
8406,"Nah I'm good thanks, I still got Netflix and t...",negative,positive,0.986793,0.003258,0.009949,0.986793
3943,"Yo, this was me last week. Good luck.",neutral,positive,0.986562,0.002895,0.010543,0.986562
9173,Great company. I added it. Let me know if you ...,neutral,positive,0.985426,0.002372,0.012202,0.985426
12632,Why thank you,neutral,positive,0.984758,0.003594,0.011648,0.984758
8950,I didn't know that! Thanks!,negative,positive,0.983339,0.005997,0.010665,0.983339
8654,Oh! Thanks for clarifying. The entire time I t...,neutral,positive,0.982243,0.004673,0.013084,0.982243
11102,"[NAME] lmaooo this made me laugh, thank you 😂",neutral,positive,0.981847,0.003643,0.014510,0.981847
10588,"I’m up here too man, having a great time so far!",neutral,positive,0.981139,0.002498,0.016363,0.981139
11198,Congrats on the sex,neutral,positive,0.980800,0.003245,0.015955,0.980800


Wrong prediction text length summary


,wrong_predictions
count,4192.000000
mean,13.835878
std,6.555584
min,1.000000
25%,8.000000
50%,14.000000
75%,19.000000
max,30.000000


Correct prediction text length summary


,correct_predictions
count,9273.000000
mean,12.988785
std,6.706554
min,1.000000
25%,7.000000
50%,13.000000
75%,18.000000
max,33.000000


In [14]:
confusion_pairs = (
    wrong_df.groupby(["true_label", "predicted_label"]).size().sort_values(ascending=False)
)
most_common_confusion_pair = (
    f"{confusion_pairs.index[0][0]} -> {confusion_pairs.index[0][1]}"
    if not confusion_pairs.empty
    else "none"
)

true_label_totals = prediction_df["true_label"].value_counts().reindex(SENTIMENT_ORDER, fill_value=0)
true_label_errors = wrong_df["true_label"].value_counts().reindex(SENTIMENT_ORDER, fill_value=0)
class_error_rates = (true_label_errors / true_label_totals.replace(0, pd.NA)).fillna(0.0)
highest_error_class = class_error_rates.idxmax() if len(class_error_rates) else "none"

wrong_mean_len = float(wrong_df["text_len"].mean()) if len(wrong_df) else 0.0
correct_mean_len = float(correct_df["text_len"].mean()) if len(correct_df) else 0.0
if wrong_mean_len < correct_mean_len - 0.5:
    length_observation = "errors are mostly shorter texts than correct predictions"
elif wrong_mean_len > correct_mean_len + 0.5:
    length_observation = "errors are mostly longer texts than correct predictions"
else:
    length_observation = "errors are spread across similar text lengths to correct predictions"

observations = []
observations.append(
    f"The most common confusion pair is {most_common_confusion_pair}, which suggests the model is blurring nearby sentiment boundaries."
)
observations.append(
    f"The highest per-class error rate is for {highest_error_class} ({class_error_rates[highest_error_class]:.2%})."
)
if not high_confidence_wrong_df.empty:
    observations.append(
        "Some failures are high-confidence mistakes, which means the model is sometimes confidently over-committing instead of expressing uncertainty."
    )
else:
    observations.append(
        "Probability columns were not available, so confidence-based failure analysis could not be expanded further."
    )

summary_lines = [
    "ERROR_ANALYSIS_SUMMARY",
    "",
    f"total errors: {len(wrong_df)}",
    f"most common confusion pair: {most_common_confusion_pair}",
    f"class with highest error rate: {highest_error_class} ({class_error_rates[highest_error_class]:.2%})",
    f"length pattern: {length_observation}",
    "observations:",
    f"- {observations[0]}",
    f"- {observations[1]}",
    f"- {observations[2]}",
]

print("\n".join(summary_lines))

ERROR_ANALYSIS_SUMMARY

total errors: 4192
most common confusion pair: negative -> neutral
class with highest error rate: negative (35.82%)
length pattern: errors are mostly longer texts than correct predictions
observations:
- The most common confusion pair is negative -> neutral, which suggests the model is blurring nearby sentiment boundaries.
- The highest per-class error rate is for negative (35.82%).
- Some failures are high-confidence mistakes, which means the model is sometimes confidently over-committing instead of expressing uncertainty.
